<a href="https://colab.research.google.com/github/SebaAcunaC/YOLO11-Vigilancia-Urbana-USACH/blob/main/Semana-4-Benchmarking/Semana4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# PROYECTO YOLO11 - SEMANA 4: BENCHMARKING DE VIDEO Y MÉTRICAS DE LATENCIA
# Objetivo: Medir el rendimiento real (FPS) en videos UHD/4K.
# ==============================================================================

# 1. SETUP E INSTALACIÓN
!pip install ultralytics -q

import os
import torch
import time
from ultralytics import YOLO

# 2. CARGA DE PESOS (REPRODUCIBILIDAD)
# Nota: Como el archivo best.pt es pesado, lo descargaremos desde tu Drive Público
# Reemplaza 'TU_ID_DE_DRIVE' por el ID real del archivo que compartiste
!wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=TU_ID_DE_DRIVE' -O best.pt

model = YOLO('best.pt')

# 3. CONFIGURACIÓN DE RUTAS LOCALES
# Para GitHub, los videos deben estar en la carpeta temporal de Colab
input_folder = 'videos_prueba'
output_folder = 'resultados_benchmarking'
os.makedirs(input_folder, exist_ok=True)
os.makedirs(output_folder, exist_ok=True)

print("Por favor, sube tus videos .mp4 a la carpeta 'videos_prueba' en el panel de la izquierda.")

# 4. PROCESAMIENTO MASIVO Y CÁLCULO DE FPS REALES
if os.path.exists(input_folder) and any(f.endswith('.mp4') for f in os.listdir(input_folder)):
    for video_name in os.listdir(input_folder):
        if video_name.endswith(".mp4"):
            print(f"\n--- Analizando: {video_name} ---")

            # Modo Stream: Crucial para no desbordar la RAM con videos 4K
            results = model.predict(
                source=os.path.join(input_folder, video_name),
                save=True,
                project=output_folder,
                name=video_name.split('.')[0],
                conf=0.5,
                stream=True
            )

            last_r = None
            for r in results:
                last_r = r

            # MÉTRICAS DE INGENIERÍA: Latencia y Rendimiento
            if last_r is not None:
                speed = last_r.speed
                t_total = speed['preprocess'] + speed['inference'] + speed['postprocess']
                fps_reales = 1000 / t_total

                print(f"Latencia Total: {t_total:.2f} ms")
                print(f"Rendimiento Real: {fps_reales:.2f} FPS")

    # 5. GENERACIÓN DE LOG AUTOMÁTICO (Sin depender de Drive)
    with open('Resumen_Rendimiento_S4_Local.txt', 'w') as f:
        f.write("RESUMEN DE RENDIMIENTO - SEMANA 4\n")
        f.write(f"Hardware: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}\n")
        f.write("-" * 40 + "\n")
        f.write(f"Latencia Promedio: {t_total:.2f} ms\n")
        f.write(f"FPS Alcanzados: {fps_reales:.2f} FPS\n")
    print("\n¡Log generado exitosamente en el entorno local!")
else:
    print("Asegúrate de subir los videos a la carpeta 'videos_prueba' antes de ejecutar.")